### Calibrate implied rates and dividend curves (OptionMetrics EOD)

In [ ]:
import logging
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.dataset as ds

from vol_risk.calibration.data_loaders import make_optionmetrics_chain
from vol_risk.calibration.transformers import apply_cutoffs, liquidity_filter
from vol_risk.models.linear import calib_linear_equity_market
from vol_risk.vol_surface.moneyness import DeltaMoneyness

logger = logging.getLogger(__name__)
plt.style.use("ggplot")

#### Configure OptionMetrics data and filters

In [ ]:
OPTIONS_DIR = Path(r"D:\\option_metrics\\parquet")
EOD_DATE = "2008-02-06 00:00:00.000000000"
TICKER = "SPX"

FILTERS = {
    "oi_min": 2,
    "bid_min": 0.01,
    "mid_min": 0.02,
    "rel_bid_ask_max": 0.5,
    "min_k_per_slice": 10,
    "delta_min": 0.01,
    "delta_max": 0.99,
}

#### Load OptionMetrics day-slice

In [ ]:
dataset = ds.dataset(str(OPTIONS_DIR), format="parquet", partitioning="hive")
df_raw = dataset.to_table(filter=ds.field("date") == EOD_DATE).to_pandas()

if df_raw.empty:
    raise RuntimeError(f"No data found for date={EOD_DATE}")

print(f"Loaded {len(df_raw):,} rows for {EOD_DATE}")

#### Convert and filter option chain

In [ ]:
raw_chain = make_optionmetrics_chain(df_raw, exclude_weeklies=True)

chain_liq = liquidity_filter(
    raw_chain,
    oi_min=FILTERS["oi_min"],
    bid_min=FILTERS["bid_min"],
    mid_min=FILTERS["mid_min"],
    rel_bid_ask_max=FILTERS["rel_bid_ask_max"],
    min_k_per_slice=FILTERS["min_k_per_slice"],
)

# Build a temporary linear market to support delta-based cutoff inversion.
lin_mkt_tmp, _, _ = calib_linear_equity_market(chain_liq)
spx_chain = apply_cutoffs(
    chain_liq,
    moneyness=DeltaMoneyness(le=lin_mkt_tmp),
    bounds=(FILTERS["delta_min"], FILTERS["delta_max"]),
)

print(f"Rows after transformer filters: {len(spx_chain):,}")
print(f"Unique expiries: {np.unique(spx_chain.expiry).size:,}")

#### Calibrate and plot per-expiry regressions

In [ ]:
n = np.unique(spx_chain.expiry).size
n_col = 3
n_row = n // n_col + int(n % n_col > 0)
fig, axes = plt.subplots(n_row, n_col, sharex=True, sharey=True, figsize=(9, 3 * n_row))

model, params, stats = calib_linear_equity_market(spx_chain, axes=np.atleast_1d(axes).ravel())

for ax in np.atleast_1d(axes).ravel()[n:]:
    ax.axis("off")

plt.tight_layout()
plt.show()
plt.close(fig)

#### Plot model output

In [ ]:
tau_granular = np.linspace(0.0001, spx_chain.tau.max() + 0.25, 100)
zero_curve_granular = model.zero_rate(tau_granular)
dvd_curve_granular = model.zero_dvd_yield(tau_granular)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(tau_granular, zero_curve_granular, label="Discounting rate", color="blue", alpha=0.7)
ax.plot(tau_granular, dvd_curve_granular, label="Dividend yield", color="orange", alpha=0.7)

stats_df = pd.DataFrame(stats).T
coeff = np.array(stats_df["coeff"].tolist(), dtype=float)
mkt_tau = stats_df["tau"].to_numpy(dtype=float)
excluded_idx = stats_df["excluded"].to_numpy(dtype=bool)

spot = spx_chain.spot
with np.errstate(invalid="ignore", divide="ignore"):
    r_mkt = -np.log(coeff[:, 1]) / mkt_tau
    q_mkt = -np.log(coeff[:, 0] / spot) / mkt_tau

valid = np.isfinite(r_mkt) & np.isfinite(q_mkt)
included = (~excluded_idx) & valid
excluded = excluded_idx & valid

ax.scatter(mkt_tau[included], r_mkt[included], color="blue", marker="o")
ax.scatter(mkt_tau[included], q_mkt[included], color="orange", marker="o")

if excluded.any():
    ax.scatter(mkt_tau[excluded], r_mkt[excluded], color="blue", marker="x")
    ax.scatter(mkt_tau[excluded], q_mkt[excluded], color="orange", marker="x")

ax.set_xlabel("Time to Maturity")
ax.set_ylabel("Implied Rates")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()

#### Print statistics

In [ ]:
pd.DataFrame(stats).T